In [1]:
import fastf1
import pandas as pd
import numpy as np

# Enable FastF1 cache
fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

In [2]:
round_number = 16
year = 2026

In [3]:
def get_long_runs(session, session_label):
    laps = session.laps.copy()
    laps = laps.pick_accurate().pick_wo_box().pick_track_status('1').copy()
    laps = laps[laps['LapTime'].notna()]
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
    laps['StartTimeSeconds'] = laps['LapStartTime'].dt.total_seconds()
    
    laps['StintLap'] = laps.groupby(['Driver', 'Stint']).cumcount() + 1
    laps = laps[laps['StintLap'] > 1]

    # Within each stint, calculate rolling minimum and discard lap times more than thresh seconds above it
    def drop_traffic(g, thresh=1.3):
        baseTime = g['LapTimeSeconds'].rolling(5, center=True, min_periods=2).min()
        return g[g['LapTimeSeconds'] < baseTime + thresh]

    session_best = laps['LapTimeSeconds'].min()
    laps = laps[laps['LapTimeSeconds'] < session_best * 1.15].copy()
    laps['session'] = session_label

    return laps

In [4]:
fp1 = fastf1.get_session(year, round_number, 'FP1')
fp1.load(telemetry=False)
fp2 = fastf1.get_session(year, round_number, 'FP2')
fp2.load(telemetry=False)
fp3 = fastf1.get_session(year, round_number, 'FP3')
fp3.load(telemetry=False)

core           INFO 	Loading data for Bahrain Grand Prix - Practice 1 [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.3]
req           

In [5]:
laps = pd.concat([
    get_long_runs(fp1, 'FP1'),
    get_long_runs(fp2, 'FP2'),
    get_long_runs(fp2, 'FP3')
], ignore_index=True)

In [6]:
# Add fuel and track evolution per session
def add_fuel_evo(laps):
    K_FUEL = 0.03 * (laps['LapTimeSeconds'].median() / 90.0)
    BURN = 1.7
    laps['fuel_kg'] = (laps.groupby(['Driver', 'Stint'])['StintLap'].transform('max') - laps['StintLap']) * BURN
    laps['evo'] = laps.groupby('session')['LapStartTime'].transform(
        lambda t: (t.dt.total_seconds() - t.dt.total_seconds().min()) / 3600.0
    )
    return laps

laps = laps.groupby('session', group_keys=False).apply(add_fuel_evo)
laps.columns

C:\Users\nanub\AppData\Local\Temp\ipykernel_13724\2296780690.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby('session', group_keys=False).apply(add_fuel_evo)


Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'LapTimeSeconds', 'StartTimeSeconds',
       'StintLap', 'session', 'fuel_kg', 'evo'],
      dtype='object')

In [7]:
import statsmodels.formula.api as smf

model = smf.rlm(
    'LapTimeSeconds ~ 0 + C(Driver) + fuel_kg + C(Compound) + C(Compound):TyreLife + evo + C(session)',
    data=laps
).fit()

rpace_raw = model.params.filter(like='C(Driver)')
rpace_raw.index = rpace_raw.index.str.extract(r'\[(.*)\]')[0].values
rpace_raw -= rpace_raw.min()

se_raw = model.bse.filter(like='C(Driver)')
se_raw.index = se_raw.index.str.extract(r'\[(.*)\]')[0].values

session_median = laps['LapTimeSeconds'].median()
practice_sigma = se_raw / session_median
rpace_raw

ALB    2.648888
ALO    1.781765
ANT    1.726330
BEA    2.881368
BOR    2.696367
BOT    2.725333
COL    2.969649
GAS    1.238923
HAD    1.303102
HAM    1.168803
HUL    2.199058
LAW    2.659351
LEC    0.649895
LIN    2.671800
NOR    0.774882
OCO    2.107194
PER    2.978594
PIA    0.176242
RUS    0.179686
SAI    3.045316
STR    3.468147
VER    0.000000
dtype: float64

In [8]:
practice_rpace = {}
for driver in rpace_raw.index:
    practice_rpace[driver] = rpace_raw[driver] / session_median

practice_rpace = pd.Series(practice_rpace)
practice_rpace = pd.concat([practice_rpace, practice_sigma], axis=1, keys=['mu', 'sigma'])
practice_rpace.sort_values(by='mu')

,mu,sigma
VER,0.000000,0.002029
PIA,0.001667,0.002049
RUS,0.001699,0.002063
LEC,0.006146,0.001849
NOR,0.007327,0.001977
HAM,0.011053,0.002065
GAS,0.011716,0.002107
HAD,0.012322,0.001887
ANT,0.016325,0.001926
ALO,0.016849,0.002089


In [9]:
practice_rpace.to_csv("practice_rpace.csv")